# 08 · Saving, loading and sharing a basis

**Who this is for:** you've built an M2C and need to keep it, hand it to a real-time controller or another tool, or come back to it next year and know how it was made.

In [1]:
import os
import tempfile

import numpy as np
import aobasis

workdir = tempfile.mkdtemp()
positions = aobasis.make_circular_actuator_grid(telescope_diameter=8.0, grid_size=16)
kl = aobasis.KLBasisGenerator(positions, fried_parameter=0.12, outer_scale=25.0, wavelength=1.65e-6)
m2c = kl.generate(80, ignore_piston=True, remove="tiptilt", normalize="rms")

## `.npz`: everything in one file

`save` writes a NumPy `.npz` file with:

- the modes and actuator positions;
- the generator's name and constructor parameters;
- the options of the `generate()` call;
- KL eigenvalues;
- the aobasis version that made it.

In [2]:
path = os.path.join(workdir, "kl_m2c.npz")
kl.save(path)

basis = aobasis.BasisGenerator.load(path)
print("type:      ", basis.basis_type)
print("version:   ", basis.aobasis_version)
print("parameters:", basis.parameters)
print("options:   ", basis.generate_options)
print("modes:     ", basis.modes.shape, " eigenvalues:", basis.eigenvalues.shape)

type:       KLBasisGenerator
version:    2.0.0
parameters: {'fried_parameter': 0.12, 'outer_scale': 25.0, 'use_gpu': False, 'r0_wavelength': 5e-07, 'wavelength': 1.65e-06}
options:    {'n_modes': 80, 'ignore_piston': True, 'orthonormalize': False, 'remove': 'tiptilt', 'normalize': 'rms'}
modes:      (172, 80)  eigenvalues: (80,)


`load` returns a `ConcreteBasis`. It doesn't rebuild the generator, but it behaves like one: `generate(n)` returns the first `n` saved modes (and stores them in `modes`, like any generator; all saved modes stay in `full_modes`), and `plot`, `report` and `save` all work.

Loading through a specific class checks the type, so a Zernike file can't be mistaken for KL:

In [3]:
first20 = basis.generate(20)
print(first20.shape)
try:
    aobasis.ZernikeBasisGenerator.load(path)
except ValueError as err:
    print("ValueError:", err)

(172, 20)
ValueError: /tmp/tmpre0i4bb4/kl_m2c.npz holds a KLBasisGenerator basis, not ZernikeBasisGenerator.


The file is plain NumPy, so no aobasis is needed to read it:

In [4]:
with np.load(path) as data:
    print(sorted(data.files))
    raw_m2c = data["modes"]

['aobasis_version', 'basis_type', 'eigenvalues', 'generate_options', 'modes', 'parameters', 'positions']


## Rebuilding a basis from its record

The saved parameters and options are enough to regenerate the basis exactly. KL modes follow a fixed sign and rotation convention, so the result is identical on any machine:

In [5]:
rebuilt = aobasis.KLBasisGenerator(basis.positions, **basis.parameters)
options = dict(basis.generate_options)  # n_modes, ignore_piston, remove, normalize, ...
again = rebuilt.generate(**options)
print("identical:", np.allclose(again, basis.full_modes))

identical: True


A `remove=` array is recorded only by its shape (`{"array_shape": [...]}`), not its values. If you removed custom arrays, keep them alongside the basis.

## FITS, for other AO software

Many AO tools exchange matrices as FITS. `save_fits` needs astropy (`pip install "aobasis[fits]"`). The primary image is the M2C, shape `(n_actuators, n_modes)` in NumPy, so `NAXIS1 = n_modes` and `NAXIS2 = n_actuators`. Extensions hold the positions and eigenvalues, and the metadata goes in the header.

In [6]:
fits_path = os.path.join(workdir, "kl_m2c.fits")
kl.save_fits(fits_path, overwrite=True)

from astropy.io import fits

with fits.open(fits_path) as hdus:
    hdus.info()
    header = hdus[0].header
    print({key: header[key] for key in ("NAXIS1", "NAXIS2", "BASIS", "AOBVER")})

same = aobasis.KLBasisGenerator.load_fits(fits_path)
print("round trip ok:", np.allclose(same.modes, m2c), same.parameters == basis.parameters)

Filename: /tmp/tmpre0i4bb4/kl_m2c.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      10   (80, 172)   float64   
  1  POSITIONS     1 ImageHDU         8   (2, 172)   float64   
  2  EIGENVAL      1 ImageHDU         7   (80,)   float64   


{'NAXIS1': 80, 'NAXIS2': 172, 'BASIS': 'KLBasisGenerator', 'AOBVER': '2.0.0'}
round trip ok: True True


Readers in other languages see the same layout. In MATLAB, for example, `fitsread("kl_m2c.fits")` returns an `n_actuators × n_modes` matrix.

For MATLAB users who prefer `.mat` files:

```python
from scipy.io import savemat
savemat("kl_m2c.mat", {"M2C": m2c, "positions": positions, "eigenvalues": kl.eigenvalues})
```

## Into a real-time controller

The M2C is just a matrix, so most controllers want it as a contiguous float32 array in actuator order:

In [7]:
m2c32 = np.ascontiguousarray(m2c, dtype=np.float32)
print(m2c32.dtype, m2c32.flags["C_CONTIGUOUS"], m2c32.shape)

float32 True (172, 80)


[pyRTC](https://github.com/jacotay7/pyRTC) builds its M2C with aobasis directly from a `basis:` section of the wavefront-corrector config (`type: kl`, `r0`, `L0`, `ignore_piston`, `normalize`, ...), so the two always agree.

## Going back: commands to modes

To read a DM state (or a measured shape) in modal terms, use the least-squares inverse. `command_to_mode_matrix` gives the C2M; `fit_coefficients` does the same for one or many command vectors, with optional regularization:

In [8]:
c2m = aobasis.command_to_mode_matrix(m2c)
true_coefficients = np.random.default_rng(0).standard_normal(80)
commands = m2c @ true_coefficients + 0.01 * np.random.default_rng(1).standard_normal(len(positions))
print("C2M recovers the coefficients to", np.abs(c2m @ commands - true_coefficients).max().round(3))
print("same with fit_coefficients:    ", np.allclose(aobasis.fit_coefficients(m2c, commands), c2m @ commands))

C2M recovers the coefficients to 0.002
same with fit_coefficients:     True


Because these modes were generated with `normalize="rms"`, each coefficient is in units of RMS phase over the actuators, and `eigenvalues` are the coefficient variances in rad² at 1.65 µm, the `wavelength` given above.

In [9]:
import shutil

shutil.rmtree(workdir)